In [0]:
df=spark.read.format('csv').option('header','true').option('inferschema','true').load('/Workspace/Users/shivshankarlkhatave@gmail.com/Pranav project 1/instagram_dataset.csv')

In [0]:
df.display()

In [0]:
df.printSchema()

In [0]:
df_sel=df.select('user_id','media_type','likes','shares','reach','hour','date').display()

In [0]:
df=df.dropDuplicates()

In [0]:
df=df.fillna({
    "likes":0,
    'comments':0,
    'shares':0,
    'saves':0
})

In [0]:
bronze=df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("my_project_11.bronze.bronze_table")

In [0]:
display(spark.table("my_project_11.bronze.bronze_table"))

In [0]:
from pyspark.sql.functions import col

silver_df=spark.read.format('delta').load("my_project_11.bronze.bronze_table")

In [0]:
%sql
select * from my_project_11.bronze.bronze_table

In [0]:
silver_df=spark.table('my_project_11.bronze.bronze_table')

In [0]:
silver_df=silver_df.withColumn("total_engagement",col('likes')+col('comments')+col('shares')+col('saves'))

In [0]:
silver_df.display()

In [0]:
silver_df.write.format('delta').mode('overwrite').saveAsTable('my_project_11.silver.silver_table')

In [0]:
silver_df.display()

In [0]:
silver_df=silver_df.withColumn('engagement_ratio',
                               col('total_engagement')/col('reach'))

In [0]:
silver_df=silver_df.withColumn('date',col('date').cast('timestamp'))

In [0]:
%sql
DROP TABLE IF EXISTS your_table;

CREATE TABLE your_table (
    date TIMESTAMP
);

In [0]:
silver_df.display()

In [0]:
gold_media=silver_df.groupBy('media_type').avg('engagement_ratio').withColumnRenamed("avg(engagement_ratio)",'avg_engagement')

In [0]:
gold_media.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("my_project_11.gold.gold_table")

In [0]:
gold_media.display()

In [0]:
gold_media.write.format('delta').mode('overwrite').saveAsTable("my_project_11.gold.gold_media")

Time Analysis

In [0]:
gold_time=silver_df.groupBy('hour','day_of_week').avg('engagement_ratio')\
    .withColumnRenamed(
    'avg(engagement_ratio)','avg_engagement'
)

In [0]:
gold_time.display()

In [0]:
gold_time.write.format('delta').mode('overwrite').saveAsTable("my_project_11.gold.gold_time")

In [0]:
gold_hashtag=silver_df.groupBy('hashtags_count').avg('engagement_ratio').withColumnRenamed('avg(engagement_ratio)','avg_engagement')

In [0]:
gold_sponsored=silver_df.groupBy('sponsored').avg('engagement_ratio').withColumnRenamed('avg(engagement_ratio)','avg_engagement')

In [0]:
gold_hashtag.display()

In [0]:
gold_sponsored.display()

In [0]:
gold_sponsored.write.format("delta").mode('overwrite').saveAsTable("my_project_11.gold.gold_sponsored")

In [0]:
gold_hashtag.write.format("delta").mode('overwrite').saveAsTable("my_project_11.gold.gold_hashtag")

In [0]:
gold_location=silver_df.groupBy("location").avg('engagement_ratio').withColumnRenamed('avg(engagement_ratio)','avg_engagement')

gold_location.write.format('delta').mode('overwrite').saveAsTable('my_project_11.gold.gold_location')

In [0]:
gold_location.display()

In [0]:
%sql
SELECT current_catalog(), current_schema();

In [0]:
%sql
select media_type, avg_engagement from my_project_11.gold.gold_media order by avg_engagement desc;

In [0]:
%sql
select hashtags_count, avg_engagement from my_project_11.gold.gold_hashtag order by avg_engagement desc;
    


In [0]:
%sql
select hour , day_of_week , avg_engagement from my_project_11.gold.gold_time order by avg_engagement desc;

In [0]:
%sql
select media_type, avg_engagement from my_project_11.gold.gold_table order by avg_engagement desc;
    
